### Imports

In [41]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [42]:
!pip install -q snntorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
import snntorch as snn
from snntorch import surrogate

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tqdm.auto import tqdm

### Configuration

In [43]:
# Configuration

SEED = 42

WINDOW = 24          # Use the previous 6 hours (24 x 15 min)
HORIZON = 1          # Predict the next 15-minute interval

BATCH_SIZE = 128
EPOCHS = 50           # increased from 30 -- give the model more room to converge
LEARNING_RATE = 1e-4

# Weighted loss: how strongly to upweight high-magnitude (peak) targets.
# 0.0 = plain MSE. Higher values push the model harder to fit spikes,
# but too high distorts predictions everywhere (see notes below).
PEAK_WEIGHT = 0.5

In [44]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)

Device: cuda


### Load Dataset

In [45]:
# Load dataset
df = pd.read_csv("/kaggle/input/datasets/kyleahmurphy/uk-electrical-load/House_2.csv")

df["Time"] = pd.to_datetime(df["Time"])
df = df.sort_values("Time")

# Keep aggregate + individual appliances
power_cols = [
    "Aggregate",
    "Appliance1", "Appliance2", "Appliance3",
    "Appliance4", "Appliance5", "Appliance6",
    "Appliance7", "Appliance8", "Appliance9"
]

df = df[["Time"] + power_cols].dropna()

# Resample to 15-minute average
df = (
    df.set_index("Time")
      .resample("15min")[power_cols]
      .mean()
      .dropna()
      .reset_index()
)

# Time-of-day features
hour = (
    df["Time"].dt.hour
    + df["Time"].dt.minute / 60
)

df["hour_sin"] = np.sin(2 * np.pi * hour / 24)
df["hour_cos"] = np.cos(2 * np.pi * hour / 24)

# Rate-of-change feature: gives the model an explicit signal that
# Aggregate is currently moving fast, which usually precedes a spike.
df["agg_diff"] = df["Aggregate"].diff().fillna(0)

print(df.shape)
print(df.head())

(45093, 14)
                 Time   Aggregate  Appliance1  Appliance2  Appliance3  \
0 2013-09-17 22:00:00  713.370968   86.580645         0.0         0.0   
1 2013-09-17 22:15:00  502.266667   75.340000         0.0         0.0   
2 2013-09-17 22:30:00   91.666667    1.006667         0.0         0.0   
3 2013-09-17 22:45:00   60.344371    1.000000         0.0         0.0   
4 2013-09-17 23:00:00   60.161074    1.000000         0.0         0.0   

   Appliance4  Appliance5  Appliance6  Appliance7  Appliance8  Appliance9  \
0         0.0         0.0         0.0         0.0         0.0         0.0   
1         0.0         0.0         0.0         0.0         0.0         0.0   
2         0.0         0.0         0.0         0.0         0.0         0.0   
3         0.0         0.0         0.0         0.0         0.0         0.0   
4         0.0         0.0         0.0         0.0         0.0         0.0   

   hour_sin  hour_cos    agg_diff  
0 -0.500000  0.866025    0.000000  
1 -0.442289  0

In [46]:
# Create t+15 target (raw Aggregate in watts, forecast horizon steps ahead)

df["Target"] = df["Aggregate"].shift(-HORIZON)

df = df.dropna().reset_index(drop=True)

print(df.head())
print(df.shape)

                 Time   Aggregate  Appliance1  Appliance2  Appliance3  \
0 2013-09-17 22:00:00  713.370968   86.580645         0.0         0.0   
1 2013-09-17 22:15:00  502.266667   75.340000         0.0         0.0   
2 2013-09-17 22:30:00   91.666667    1.006667         0.0         0.0   
3 2013-09-17 22:45:00   60.344371    1.000000         0.0         0.0   
4 2013-09-17 23:00:00   60.161074    1.000000         0.0         0.0   

   Appliance4  Appliance5  Appliance6  Appliance7  Appliance8  Appliance9  \
0         0.0         0.0         0.0         0.0         0.0         0.0   
1         0.0         0.0         0.0         0.0         0.0         0.0   
2         0.0         0.0         0.0         0.0         0.0         0.0   
3         0.0         0.0         0.0         0.0         0.0         0.0   
4         0.0         0.0         0.0         0.0         0.0         0.0   

   hour_sin  hour_cos    agg_diff      Target  
0 -0.500000  0.866025    0.000000  502.266667  
1 

### Preprocessing

In [47]:
# Chronological split

n = len(df)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df.iloc[:train_end].copy()
val_df   = df.iloc[train_end:val_end].copy()
test_df  = df.iloc[val_end:].copy()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nTrain:", train_df["Time"].min(), "->", train_df["Time"].max())
print("Val:  ", val_df["Time"].min(), "->", val_df["Time"].max())
print("Test: ", test_df["Time"].min(), "->", test_df["Time"].max())

Train: (31564, 15)
Validation: (6764, 15)
Test: (6764, 15)

Train: 2013-09-17 22:00:00 -> 2014-12-21 03:30:00
Val:   2014-12-21 03:45:00 -> 2015-03-01 16:00:00
Test:  2015-03-01 16:15:00 -> 2015-05-28 07:45:00


In [48]:
power_cols = [
    "Aggregate",
    "Appliance1", "Appliance2", "Appliance3",
    "Appliance4", "Appliance5", "Appliance6",
    "Appliance7", "Appliance8", "Appliance9"
]

scaler = StandardScaler()

train_scaled = scaler.fit_transform(train_df[power_cols])
val_scaled = scaler.transform(val_df[power_cols])
test_scaled = scaler.transform(test_df[power_cols])

# agg_diff isn't in power_cols, so it needs its own scaler
diff_scaler = StandardScaler()
train_df["agg_diff_scaled"] = diff_scaler.fit_transform(train_df[["agg_diff"]])
val_df["agg_diff_scaled"] = diff_scaler.transform(val_df[["agg_diff"]])
test_df["agg_diff_scaled"] = diff_scaler.transform(test_df[["agg_diff"]])

print("train_scaled:", train_scaled.shape)

train_scaled: (31564, 10)


In [49]:
# Create input sequences
# Features used as model input: Aggregate + 9 appliances + time-of-day + rate-of-change
feature_cols = [
    "Aggregate",
    "Appliance1", "Appliance2", "Appliance3",
    "Appliance4", "Appliance5", "Appliance6",
    "Appliance7", "Appliance8", "Appliance9",
    "hour_sin",
    "hour_cos",
    "agg_diff_scaled"
]

def create_sequences(scaled_power, df_split, window=WINDOW):
    X = []
    y = []

    hour_sin = df_split["hour_sin"].values
    hour_cos = df_split["hour_cos"].values
    agg_diff = df_split["agg_diff_scaled"].values

    for i in range(len(scaled_power) - window):
        sequence = np.column_stack([
            scaled_power[i:i + window],           # Aggregate + Appliance1-9
            hour_sin[i:i + window],
            hour_cos[i:i + window],
            agg_diff[i:i + window]
        ])

        X.append(sequence)
        y.append(scaled_power[i + window, 0])     # Future Aggregate (scaled)

    return np.array(X), np.array(y).reshape(-1, 1)

X_train, y_train = create_sequences(train_scaled, train_df)
X_val, y_val = create_sequences(val_scaled, val_df)
X_test, y_test = create_sequences(test_scaled, test_df)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

X_train: (31540, 24, 13)
y_train: (31540, 1)


In [ ]:
X_train, y_train = create_sequences(train_scaled, train_df, WINDOW)
X_val, y_val = create_sequences(val_scaled, val_df, WINDOW)
X_test, y_test = create_sequences(test_scaled, test_df, WINDOW)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:  ", X_val.shape, "y_val:  ", y_val.shape)
print("X_test: ", X_test.shape, "y_test: ", y_test.shape)

In [50]:
# Convert to PyTorch tensors

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)

X_val = torch.tensor(X_val, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.float32)

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

print(X_train.shape, y_train.shape)

torch.Size([31540, 24, 13]) torch.Size([31540, 1])


In [51]:
# Create DataLoaders

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(X_val, y_val),
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    TensorDataset(X_test, y_test),
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 247
Validation batches: 53
Test batches: 53


### Neural Network

# V5 — Non-SNN Energy Forecasting Baselines

For now, **forget SNN/RSNN completely**.

We first establish how predictable the target is using the exact same forecasting setup:

**previous energy window + hour sin/cos + appliance energy → next 15-minute aggregate energy**

Experiments:
1. Target distribution / histogram
2. Persistence
3. Linear Regression
4. Random Forest
5. XGBoost
6. MLP
7. LSTM
8. GRU
9. Peak-specific metrics
10. Actual-vs-predicted plots

## 1. Baseline evaluation

In [ ]:
def evaluate(actual, pred, name):
    actual = np.asarray(actual).ravel()
    pred = np.asarray(pred).ravel()

    mae = mean_absolute_error(actual, pred)
    rmse = np.sqrt(mean_squared_error(actual, pred))
    r2 = r2_score(actual, pred)

    p90 = np.percentile(actual, 90)
    mask = actual >= p90
    peak_mae = mean_absolute_error(actual[mask], pred[mask])
    peak_rmse = np.sqrt(mean_squared_error(actual[mask], pred[mask]))

    print(f"\n{name}")
    print(f"MAE       : {mae:.2f} W")
    print(f"RMSE      : {rmse:.2f} W")
    print(f"R²        : {r2:.4f}")
    print(f"Peak MAE  : {peak_mae:.2f} W")
    print(f"Peak RMSE : {peak_rmse:.2f} W")
    print(f"Actual max: {actual.max():.2f} W")
    print(f"Pred max  : {pred.max():.2f} W")
    print(f"Peak ratio: {pred.max()/actual.max():.4f}")

    return {
        "Model":name, "MAE":mae, "RMSE":rmse, "R2":r2,
        "Peak_MAE":peak_mae, "Peak_RMSE":peak_rmse,
        "Actual_Max":actual.max(), "Pred_Max":pred.max(),
        "Peak_Ratio":pred.max()/actual.max()
    }

results=[]

## 2. Persistence baseline

In [ ]:
# Predict the next interval using the current aggregate.
# This is the minimum useful baseline.

actual = test_df["Aggregate"].values[1:]
pred = test_df["Aggregate"].values[:-1]

results.append(evaluate(actual, pred, "Persistence"))

## 3. Flatten the same windows for tabular models

In [ ]:
X_train_flat = X_train.reshape(len(X_train), -1)
X_val_flat   = X_val.reshape(len(X_val), -1)
X_test_flat  = X_test.reshape(len(X_test), -1)

y_train_flat = y_train.ravel()
y_val_flat   = y_val.ravel()
y_test_flat  = y_test.ravel()

print(X_train_flat.shape, X_val_flat.shape, X_test_flat.shape)

## 4. Linear Regression

In [ ]:
linear = LinearRegression()
linear.fit(X_train_flat, y_train_flat)

linear_pred = linear.predict(X_test_flat)
results.append(evaluate(y_test_flat, linear_pred, "Linear Regression"))

## 5. Random Forest

In [ ]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=SEED
)

rf.fit(X_train_flat, y_train_flat)
rf_pred = rf.predict(X_test_flat)

results.append(evaluate(y_test_flat, rf_pred, "Random Forest"))

## 6. XGBoost

In [ ]:
# Install if necessary:
# !pip install -q xgboost

from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=SEED,
    n_jobs=-1
)

xgb.fit(
    X_train_flat,
    y_train_flat,
    eval_set=[(X_val_flat, y_val_flat)],
    verbose=False
)

xgb_pred = xgb.predict(X_test_flat)
results.append(evaluate(y_test_flat, xgb_pred, "XGBoost"))

## 7. MLP

In [ ]:
class MLP(nn.Module):
    def __init__(self, window, features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(window * features, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x.flatten(1))


def train_torch(model, train_loader, val_loader, epochs=50, lr=1e-3):
    model = model.to(DEVICE)
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.MSELoss()
    train_hist, val_hist = [], []

    for epoch in range(epochs):
        model.train()
        total=0

        for xb,yb in train_loader:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE)
            opt.zero_grad()
            loss=loss_fn(model(xb),yb)
            loss.backward()
            opt.step()
            total += loss.item()

        train_hist.append(total/len(train_loader))

        model.eval()
        total=0
        with torch.no_grad():
            for xb,yb in val_loader:
                xb,yb=xb.to(DEVICE),yb.to(DEVICE)
                total += loss_fn(model(xb),yb).item()

        val_hist.append(total/len(val_loader))

        if epoch==0 or (epoch+1)%10==0:
            print(
                f"Epoch {epoch+1:02d}/{epochs} | "
                f"Train {train_hist[-1]:.4f} | Val {val_hist[-1]:.4f}"
            )

    return model,train_hist,val_hist


mlp=MLP(WINDOW,X_train.shape[-1])
mlp,mlp_train,mlp_val=train_torch(
    mlp,train_loader,val_loader,EPOCHS,1e-3
)

mlp.eval()
mlp_pred=[]

with torch.no_grad():
    for xb,_ in test_loader:
        mlp_pred.append(mlp(xb.to(DEVICE)).cpu().numpy())

mlp_pred=np.concatenate(mlp_pred).ravel()
mlp_result=evaluate(y_test_flat,mlp_pred,"MLP")
results.append(mlp_result)

plt.figure(figsize=(8,4))
plt.plot(mlp_train,label="Train")
plt.plot(mlp_val,label="Validation")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("MLP Loss")
plt.legend()
plt.grid(True)
plt.show()

## 8. LSTM

In [ ]:
class RNNForecaster(nn.Module):
    def __init__(self,input_size,hidden_size=64,kind="LSTM"):
        super().__init__()
        self.kind=kind

        if kind=="LSTM":
            self.rnn=nn.LSTM(input_size,hidden_size,batch_first=True)
        else:
            self.rnn=nn.GRU(input_size,hidden_size,batch_first=True)

        self.fc=nn.Linear(hidden_size,1)

    def forward(self,x):
        out,_=self.rnn(x)
        return self.fc(out[:,-1,:])


def run_rnn(kind):
    model=RNNForecaster(X_train.shape[-1],64,kind)
    model,train_hist,val_hist=train_torch(
        model,train_loader,val_loader,EPOCHS,1e-3
    )

    model.eval()
    preds=[]

    with torch.no_grad():
        for xb,_ in test_loader:
            preds.append(model(xb.to(DEVICE)).cpu().numpy())

    pred=np.concatenate(preds).ravel()

    result=evaluate(y_test_flat,pred,kind)

    plt.figure(figsize=(8,4))
    plt.plot(train_hist,label="Train")
    plt.plot(val_hist,label="Validation")
    plt.xlabel("Epoch")
    plt.ylabel("MSE")
    plt.title(f"{kind} Loss")
    plt.legend()
    plt.grid(True)
    plt.show()

    return model,pred,result


lstm,lstm_pred,lstm_result=run_rnn("LSTM")
results.append(lstm_result)

## 9. GRU

In [ ]:
gru,gru_pred,gru_result=run_rnn("GRU")
results.append(gru_result)

## 10. Compare everything

In [ ]:
results_df=pd.DataFrame(results)

results_df=results_df[
    [
        "Model","MAE","RMSE","R2",
        "Peak_MAE","Peak_RMSE",
        "Actual_Max","Pred_Max","Peak_Ratio"
    ]
].sort_values("RMSE").reset_index(drop=True)

display(results_df)

## 11. Prediction plots

In [ ]:
def plot_predictions(actual,pred,title,points=192):
    points=min(points,len(actual))

    plt.figure(figsize=(13,5))
    plt.plot(actual[:points],label="Actual")
    plt.plot(pred[:points],label="Predicted")
    plt.xlabel("15-minute interval")
    plt.ylabel("Aggregate Power (W)")
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()


plot_predictions(y_test_flat,lstm_pred,"LSTM: Actual vs Predicted")
plot_predictions(y_test_flat,gru_pred,"GRU: Actual vs Predicted")
plot_predictions(y_test_flat,mlp_pred,"MLP: Actual vs Predicted")

## 12. Peak-focused comparison

In [ ]:
p90=np.percentile(y_test_flat,90)
mask=y_test_flat>=p90

plt.figure(figsize=(13,5))
plt.plot(np.where(mask)[0],y_test_flat[mask],"o",label="Actual")
plt.plot(np.where(mask)[0],lstm_pred[mask],"o",label="LSTM")
plt.plot(np.where(mask)[0],gru_pred[mask],"o",label="GRU")
plt.xlabel("Test sample")
plt.ylabel("Aggregate Power (W)")
plt.title("P90+ High-Energy Events")
plt.legend()
plt.grid(True)
plt.show()

## 13. How to interpret the results

Do not return to SNN tuning yet.

First determine:

- Does persistence already explain most of the short-term variation?
- Does MLP beat the recurrent models?
- Does LSTM/GRU beat MLP?
- Do Random Forest/XGBoost beat the neural models?
- Which model has the best **Peak MAE / Peak RMSE**?
- Which model has the best **Peak Ratio**?

If conventional models also underestimate peaks, the problem is probably not SNN-specific.

If conventional models capture the peaks much better, we have a clear target for improving the eventual SNN.